# Học máy ứng dụng – Bài 3: Naive Bayes


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_csv("/content/tin_nhan.csv")
print("Kich thuoc bang:", df.shape)
print()
print("So tin theo nhan (1 la rac, 0 la thuong):")
print(df["la_rac"].value_counts())
print()
print("Ty le tin rac:", round(df["la_rac"].mean(), 4))
print()

print("Ba tin nhan dau tien:")
for i in range(3):
    print(f"  [{df.la_rac[i]}] {df.noi_dung[i]}")
print()

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])

print("Kich thuoc bang dem tu:", X.shape)
print("So tu trong tu dien:", len(vec.vocabulary_))
print()

# Dem tong so lan xuat hien cua tung tu, tach theo hai lop
dem = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
dem["la_rac"] = df["la_rac"]
tong = dem.groupby("la_rac").sum().T
tong.columns = ["thuong", "rac"]

print("Muoi tu xuat hien nhieu nhat trong tin rac:")
print(tong.sort_values("rac", ascending=False).head(10))
print()
print("Muoi tu xuat hien nhieu nhat trong tin thuong:")
print(tong.sort_values("thuong", ascending=False).head(10))


Kich thuoc bang: (300, 2)

So tin theo nhan (1 la rac, 0 la thuong):
la_rac
0    180
1    120
Name: count, dtype: int64

Ty le tin rac: 0.4

Ba tin nhan dau tien:
  [0] cáo gửi cáo thầy nhóm cáo gửi có cho
  [1] miễn giảm ngay hạn miễn tay nhé thưởng cho nhanh
  [0] cho bạn mai gửi nhóm bài chiều

Kich thuoc bang dem tu: (300, 40)
So tu trong tu dien: 40

Muoi tu xuat hien nhieu nhat trong tin rac:
       thuong  rac
tay         3   48
ngay        4   47
nhanh       9   44
chót        6   43
giảm        6   42
lúc        39   42
quà         8   41
click       5   40
phí         6   40
miễn        4   39

Muoi tu xuat hien nhieu nhat trong tin thuong:
       thuong  rac
thầy       74    7
bài        73    4
cáo        72    8
chiều      68    8
ạ          65    7
tài        64   11
tập        63    6
gửi        61   10
lớp        60    2
nhóm       58    7


## Ví dụ 3: Bộ lọc tin nhắn rác với MultinomialNB

Huấn luyện bộ lọc, in ma trận nhầm lẫn, accuracy, precision, recall và thử trên vài tin tự gõ.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print("So tin de hoc      :", X_train.shape[0])
print("So tin de kiem tra :", X_test.shape[0])
print()

mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X_train, y_train)
y_pred = mo_hinh.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("Ma tran nham lan")
print(f"  TN = {tn:2d}  thu thuong duoc giu dung")
print(f"  FP = {fp:2d}  thu thuong bi xoa nham")
print(f"  FN = {fn:2d}  thu rac lot luoi")
print(f"  TP = {tp:2d}  thu rac bi chan dung")
print()

print(f"Accuracy  = {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision = {precision_score(y_test, y_pred):.4f}")
print(f"Recall    = {recall_score(y_test, y_pred):.4f}")
print()

# Thu bo loc tren vai tin nhan tu go
tin_moi = [
    "trung thuong khuyen mai nhanh tay click ngay",
    "hop nhom bao cao chieu mai phong 305",
    "ban gui giup em tai lieu bai tap nhe",
]
X_moi = vec.transform(tin_moi)          # chi transform, khong fit_transform
p_rac = mo_hinh.predict_proba(X_moi)[:, 1]

print("Thu tren ba tin nhan tu go:")
for tin, p in zip(tin_moi, p_rac):
    nhan = "RAC" if p >= 0.5 else "thuong"
    print(f"  [{nhan:6s}] p = {p:.4f} <- {tin}")


So tin de hoc      : 225
So tin de kiem tra : 75

Ma tran nham lan
  TN = 45  thu thuong duoc giu dung
  FP =  0  thu thuong bi xoa nham
  FN =  4  thu rac lot luoi
  TP = 26  thu rac bi chan dung

Accuracy  = 0.9467
Precision = 1.0000
Recall    = 0.8667

Thu tren ba tin nhan tu go:
  [RAC   ] p = 0.9992 <- trung thuong khuyen mai nhanh tay click ngay
  [thuong] p = 0.0983 <- hop nhom bao cao chieu mai phong 305
  [thuong] p = 0.0719 <- ban gui giup em tai lieu bai tap nhe


## Ví dụ 4: Vấn đề số không và làm trơn Laplace

Tự cài Naive Bayes (không dùng scikit-learn) trên kho 6 tin nhắn để thấy mô hình gãy khi α = 0.

In [ ]:
from collections import Counter

# Kho tin nhan ti hon, viet thang vao ma cho de theo doi
kho = [
    ("trung thuong ngay", 1),
    ("khuyen mai giam gia", 1),
    ("trung thuong click", 1),
    ("hop nhom chieu mai", 0),
    ("bai tap nhom nop mai", 0),
    ("thay gui tai lieu", 0),
]

tu_dien = sorted({t for cau, _ in kho for t in cau.split()})
V = len(tu_dien)
print(f"Tu dien co {V} tu:")
for i in range(0, V, 9):
    print("  " + " ".join(tu_dien[i:i + 9]))
print()

# Dem so lan moi tu xuat hien trong tung lop
dem = {0: Counter(), 1: Counter()}
tong = {0: 0, 1: 0}
for cau, nhan in kho:
    for t in cau.split():
        dem[nhan][t] += 1
        tong[nhan] += 1

print("So lan xuat hien cua tung tu")
print(f"  {'tu':12s} {'rac':>5s} {'thuong':>7s}")
for t in tu_dien:
    print(f"  {t:12s} {dem[1][t]:5d} {dem[0][t]:7d}")
print()


def xac_suat(tu, nhan, alpha):
    """P(tu | nhan) co lam tron Laplace voi he so alpha."""
    return (dem[nhan][tu] + alpha) / (tong[nhan] + alpha * V)


def diem(cau, nhan, alpha):
    """Tich cua xac suat tien nghiem va cac xac suat tu."""
    p = 0.5  # hai lop can bang nen tien nghiem bang nhau
    for t in cau.split():
        if t in tu_dien:
            p *= xac_suat(t, nhan, alpha)
    return p


cau_thu = "trung thuong bai tap"
print(f'Tin nhan thu: "{cau_thu}"')
print("Tu 'bai' chi xuat hien o lop thuong, chua bao gio o lop rac.")
print()

for alpha in [0.0, 1.0]:
    d_rac = diem(cau_thu, 1, alpha)
    d_thuong = diem(cau_thu, 0, alpha)
    ket = "rac" if d_rac > d_thuong else "thuong"
    print(f"alpha = {alpha}")
    print(f"  diem lop rac    = {d_rac:.10f}")
    print(f"  diem lop thuong = {d_thuong:.10f}")
    print(f"  ket luan        = {ket}")
    print()

print("Khong lam tron, CA HAI diem deu bang 0 nen mo hinh te liet.")
print("Lop rac bang 0 vi tu 'bai' chua tung xuat hien o lop rac.")
print("Lop thuong bang 0 vi tu 'trung' chua tung xuat hien o lop thuong.")
print("Co lam tron, diem lop rac lon hon va mo hinh ket luan dung.")


Tu dien co 18 tu:
  bai chieu click gia giam gui hop khuyen lieu
  mai ngay nhom nop tai tap thay thuong trung

So lan xuat hien cua tung tu
  tu             rac  thuong
  bai              0       1
  chieu            0       1
  click            1       0
  gia              1       0
  giam             1       0
  gui              0       1
  hop              0       1
  khuyen           1       0
  lieu             0       1
  mai              1       2
  ngay             1       0
  nhom             0       2
  nop              0       1
  tai              0       1
  tap              0       1
  thay             0       1
  thuong           2       0
  trung            2       0

Tin nhan thu: "trung thuong bai tap"
Tu 'bai' chi xuat hien o lop thuong, chua bao gio o lop rac.

alpha = 0.0
  diem lop rac    = 0.0000000000
  diem lop thuong = 0.0000000000
  ket luan        = thuong

alpha = 1.0
  diem lop rac    = 0.0000073212
  diem lop thuong = 0.0000021656
  ket luan        = rac


## Ví dụ 5: Naive Bayes cho số liệu liên tục (GaussianNB)

So sánh GaussianNB với hồi quy logistic trên `data/sinh_vien.csv`, cùng cách chia dữ liệu.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

df = pd.read_csv("/content/sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Hai mo hinh, cung du lieu, cung cach chia, nen so sanh moi cong bang
for ten, mo_hinh in [("Naive Bayes     ", GaussianNB()),
                     ("Hoi quy logistic", LogisticRegression())]:
    mo_hinh.fit(X_train, y_train)
    acc = accuracy_score(y_test, mo_hinh.predict(X_test))
    print(f"{ten}: accuracy = {acc:.4f}")
print()

gnb = GaussianNB().fit(X_train, y_train)

# theta_ la trung binh, var_ la phuong sai, moi dong mot lop
print("Nhung gi Naive Bayes hoc duoc")
print(f"  {'':16s} {'gio_on':>10s} {'diem_giua_ky':>14s}")
for i, ten_lop in enumerate(["Lop 0 (rot) ", "Lop 1 (qua) "]):
    tb = gnb.theta_[i]
    dl = gnb.var_[i] ** 0.5
    print(f"  {ten_lop}    trung binh {tb[0]:10.2f} {tb[1]:14.2f}")
    print(f"  {'':12s}    do lech    {dl[0]:10.2f} {dl[1]:14.2f}")
print()

ty_le = ", ".join(f"{float(v):.4f}" for v in gnb.class_prior_)
print(f"Ty le tung lop trong tap hoc: {ty_le}")
print()

# Du doan cho vai sinh vien gia dinh
ban_moi = pd.DataFrame({"gio_on": [5, 13, 20, 28],
                        "diem_giua_ky": [4.0, 6.0, 7.0, 9.0]})
p = gnb.predict_proba(ban_moi)[:, 1]
print("Du doan cho bon ban moi:")
print(f"  {'gio on':>8s} {'diem GK':>9s} {'P(qua mon)':>12s}")
for (_, r), pi in zip(ban_moi.iterrows(), p):
    print(f"  {r.gio_on:8.0f} {r.diem_giua_ky:9.1f} {pi:12.4f}")


Naive Bayes     : accuracy = 0.9000
Hoi quy logistic: accuracy = 0.9000

Nhung gi Naive Bayes hoc duoc
                       gio_on   diem_giua_ky
  Lop 0 (rot)     trung binh       8.18           3.99
                  do lech          5.29           1.92
  Lop 1 (qua)     trung binh      20.03           6.70
                  do lech          5.81           1.50

Ty le tung lop trong tap hoc: 0.4111, 0.5889

Du doan cho bon ban moi:
    gio on   diem GK   P(qua mon)
         5       4.0       0.0137
        13       6.0       0.6550
        20       7.0       0.9855
        28       9.0       0.9999


---
# Bài tập

## Bài tập 1: Bayes với ba nguyên nhân

Một lớp có ba nhóm sinh viên: 50% học đều, 30% học dồn, 20% ít học. Tỷ lệ qua môn của ba nhóm lần lượt là 0.95, 0.70 và 0.30.

Gặp một bạn đã qua môn. Tính xác suất bạn đó thuộc từng nhóm. Ba con số cộng lại phải bằng 1, hãy dùng điều đó để tự kiểm tra.

*Gợi ý: mở rộng hàm `bayes` trong ví dụ 1 cho ba nhánh thay vì hai.*

In [ ]:
ten = ["hoc deu", "hoc don", "it hoc"]
tien_nghiem = [0.50, 0.30, 0.20]   # P(nhom)
kha_nang = [0.95, 0.70, 0.30]      # P(qua mon | nhom)

tu_so = [t * k for t, k in zip(tien_nghiem, kha_nang)]
bang_chung = sum(tu_so)            # P(qua mon), xac suat toan phan
hau_nghiem = [x / bang_chung for x in tu_so]

print(f"P(qua mon) = {bang_chung:.4f}")
for t, h in zip(ten, hau_nghiem):
    print(f"  P({t} | qua mon) = {h:.4f}")
print(f"Tong ba xac suat = {sum(hau_nghiem):.4f} (phai bang 1)")


P(qua mon) = 0.7450
  P(hoc deu | qua mon) = 0.6376
  P(hoc don | qua mon) = 0.2819
  P(it hoc | qua mon) = 0.0805
Tong ba xac suat = 1.0000 (phai bang 1)


## Bài tập 2: Từ nào tố cáo mạnh nhất

Huấn luyện `MultinomialNB` trên toàn bộ 300 tin nhắn. Dùng thuộc tính `feature_log_prob_` để tính hiệu giữa hai lớp cho từng từ, rồi in ra 5 từ tố cáo tin rác mạnh nhất và 5 từ cho thấy tin thường mạnh nhất.

*Gợi ý: `feature_log_prob_` là mảng hai dòng, dòng 0 cho lớp 0 và dòng 1 cho lớp 1. Lấy `mo_hinh.feature_log_prob_[1] - mo_hinh.feature_log_prob_[0]` rồi sắp xếp.*

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
mo_hinh = MultinomialNB(alpha=1.0).fit(X, df["la_rac"])

tu = vec.get_feature_names_out()
chenh = mo_hinh.feature_log_prob_[1] - mo_hinh.feature_log_prob_[0]
bang = pd.Series(chenh, index=tu).sort_values()

print("5 tu to cao tin RAC manh nhat:")
print(bang.tail(5)[::-1].round(3))
print()
print("5 tu cho thay tin THUONG manh nhat:")
print(bang.head(5).round(3))


5 tu to cao tin RAC manh nhat:
mãi      3.203
tay      2.906
ngay     2.662
miễn     2.480
trúng    2.374
dtype: float64

5 tu cho thay tin THUONG manh nhat:
lớp     -2.612
em      -2.562
bài     -2.295
báo     -2.274
phòng   -2.274
dtype: float64


## Bài tập 3: Đổi tham số làm trơn

Chạy lại bộ lọc với `alpha` lần lượt bằng 0.001, 0.1, 1.0, 10.0 và 100.0. Với mỗi giá trị, in ra accuracy, precision và recall trên tập kiểm tra.

Nhận xét một câu: alpha lớn quá thì chuyện gì xảy ra, và vì sao?

*Gợi ý cho phần giải thích: alpha rất lớn làm mọi xác suất từ xích lại gần nhau.*

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y)

print(f"{'alpha':>8s} {'accuracy':>9s} {'precision':>10s} {'recall':>7s}")
for alpha in [0.001, 0.1, 1.0, 10.0, 100.0]:
    y_pred = MultinomialNB(alpha=alpha).fit(X_train, y_train).predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    print(f"{alpha:8g} {acc:9.4f} {pre:10.4f} {rec:7.4f}")

# Nhan xet: alpha qua lon thi alpha*V ap dao so dem that, moi tu co xac suat
# gan nhu bang nhau o hai lop -> mo hinh mat kha nang phan biet,
# chi con dua vao tien nghiem (lop dong hon thang).


## Bài tập 4: Tự viết bộ lọc không dùng thư viện

Dựa vào ví dụ 4, viết một bộ lọc hoàn chỉnh cho toàn bộ 300 tin nhắn. Tự đếm từ, tự làm trơn, tự tính điểm cho hai lớp bằng **tổng loga** thay vì tích.

Chia dữ liệu bằng `train_test_split` với `random_state=17` rồi so accuracy của bộ lọc tự viết với `MultinomialNB`. Hai con số phải rất gần nhau. Nếu lệch nhiều thì có lỗi ở đâu đó, hãy tìm ra.

In [ ]:
import math
from collections import Counter

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

train, test = train_test_split(df, test_size=0.25, random_state=17,
                               stratify=df["la_rac"])

# Tu dien lap tren toan bo kho (giong d3, vec.fit_transform tren df)
tu_dien = sorted({t for c in df["noi_dung"] for t in c.split()})
V = len(tu_dien)
alpha = 1.0

dem = {0: Counter(), 1: Counter()}
tong = {0: 0, 1: 0}
so_tin = {0: 0, 1: 0}
for cau, nhan in zip(train["noi_dung"], train["la_rac"]):
    so_tin[nhan] += 1
    for t in cau.split():
        dem[nhan][t] += 1
        tong[nhan] += 1

log_tien = {k: math.log(so_tin[k] / len(train)) for k in (0, 1)}
log_tu = {k: {t: math.log((dem[k][t] + alpha) / (tong[k] + alpha * V))
              for t in tu_dien} for k in (0, 1)}


def du_doan(cau):
    diem = {}
    for k in (0, 1):
        diem[k] = log_tien[k] + sum(log_tu[k][t] for t in cau.split()
                                    if t in log_tu[k])
    return 1 if diem[1] > diem[0] else 0


y_tu_viet = [du_doan(c) for c in test["noi_dung"]]
acc_tu_viet = accuracy_score(test["la_rac"], y_tu_viet)

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
X_tr, X_te, y_tr, y_te = train_test_split(X, df["la_rac"], test_size=0.25,
                                          random_state=17, stratify=df["la_rac"])
acc_thu_vien = accuracy_score(y_te, MultinomialNB(alpha=1.0).fit(X_tr, y_tr).predict(X_te))

print(f"Tu viet   : accuracy = {acc_tu_viet:.4f}")
print(f"MultinomialNB: accuracy = {acc_thu_vien:.4f}")


## Bài tập 5: Chọn sai biến thể Naive Bayes

Chạy `MultinomialNB` trên bộ dữ liệu sinh viên với hai cột `gio_on` và `diem_giua_ky`. In ra accuracy rồi so với `GaussianNB` ở ví dụ 5.

Chương trình có báo lỗi không? Con số accuracy ra bao nhiêu? Giải thích vì sao dùng `MultinomialNB` ở đây là sai về nguyên tắc, dù nó vẫn chạy.

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB

X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y)

for ten, mh in [("GaussianNB   ", GaussianNB()), ("MultinomialNB", MultinomialNB())]:
    mh.fit(X_train, y_train)
    print(f"{ten}: accuracy = {accuracy_score(y_test, mh.predict(X_test)):.4f}")

# Nhan xet: MultinomialNB khong bao loi vi gio_on, diem_giua_ky deu >= 0,
# nhung no coi chung la "so lan xuat hien" (du lieu dem). Gio on va diem la
# so thuc lien tuc, nen mo hinh tinh xac suat tren gia dinh sai -> sai nguyen tac.


## Bài tập 6: Lớp hiếm và xác suất tiên nghiệm

Từ bộ tin nhắn, tạo một bộ mới mất cân bằng nặng: giữ toàn bộ 180 tin thường nhưng chỉ lấy 20 tin rác đầu tiên. Huấn luyện lại và chấm điểm.

In ra accuracy và recall của lớp rác. Nhận xét: accuracy có còn phản ánh đúng chất lượng bộ lọc không?

*Gợi ý: dùng `recall_score` với `pos_label=1`, và nhớ lại cái bẫy accuracy ở bài 2.*

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

lech = pd.concat([df[df["la_rac"] == 0],
                  df[df["la_rac"] == 1].head(20)]).reset_index(drop=True)
print(lech["la_rac"].value_counts())

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(lech["noi_dung"])
y = lech["la_rac"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y)

y_pred = MultinomialNB(alpha=1.0).fit(X_train, y_train).predict(X_test)
print(f"Accuracy      = {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall (rac)  = {recall_score(y_test, y_pred, pos_label=1):.4f}")

# Nhan xet: tien nghiem cua lop rac chi ~10%, nen mo hinh nghieng ve "thuong".
# Accuracy van cao (chi can doan toan "thuong" da ~90%) nhung recall lop rac
# thap -> accuracy khong con phan anh dung chat luong bo loc.
